# Exploratory Data Analysis - Chicago Crimes (2001-Present)

**Objective:** Understand crime patterns, temporal trends, geospatial distributions, and class balance.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('ChicagoCrimes_EDA').enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print('Spark session ready')

In [ ]:
df = spark.sql('SELECT * FROM team2.crimes_features')
print('Total records:', df.count())
df.printSchema()

## 1. Arrest Distribution (Class Balance)

In [ ]:
df.groupBy('arrest').count().show()
df.groupBy('arrest').count().toPandas().plot.bar(x='arrest', y='count')

## 2. Temporal Patterns - Crimes by Year

In [ ]:
yearly = df.groupBy('year').count().orderBy('year').toPandas()
yearly.plot(x='year', y='count', figsize=(14,5), title='Crime Count by Year')

## 3. Crimes by Hour of Day

In [ ]:
hourly = df.groupBy('hour_of_day').count().orderBy('hour_of_day').toPandas()
hourly.plot(x='hour_of_day', y='count', kind='bar', figsize=(12,5), title='Crimes by Hour')

## 4. Top Crime Types

In [ ]:
df.groupBy('primary_type').count().orderBy(desc('count')).show(15, truncate=False)

## 5. Geospatial Distribution (sample)

In [ ]:
geo = df.filter(col('latitude').isNotNull()).select('latitude','longitude','primary_type').sample(False, 0.01).toPandas()
geo.plot(kind='scatter', x='longitude', y='latitude', alpha=0.1, figsize=(10,8), s=1)

## 6. Arrest Rate by District

In [ ]:
from pyspark.sql.functions import round

district_arrest = df.groupBy('district').agg(
    count('*').alias('total'),
    sum(when(col('arrest')==True, 1).otherwise(0)).alias('arrests')
).withColumn('arrest_rate', round(col('arrests')/col('total')*100, 2))
.orderBy(desc('arrest_rate'))

district_arrest.show(25)